In [ ]:
import numpy as np
import scipy
import skimage

In [ ]:
filename = 'Leopard'
flow_direction = 'z'
raw_dimension = 1000
dtype = np.float32

## Flow between pores

In [ ]:
pores_img = np.fromfile(f"../raw_data/{filename}_pores.raw", dtype=np.int32, sep="")
pores_img = pores_img.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
vx = np.fromfile(f'../raw_data/{filename}_{flow_direction}_ux.raw', dtype=dtype, sep='')
vx = vx.reshape(raw_dimension, raw_dimension, raw_dimension)
vy = np.fromfile(f'../raw_data/{filename}_{flow_direction}_uy.raw', dtype=dtype, sep='')
vy = vy.reshape(raw_dimension, raw_dimension, raw_dimension)
vz = np.fromfile(f'../raw_data/{filename}_{flow_direction}_uz.raw', dtype=dtype, sep='')
vz = vz.reshape(raw_dimension, raw_dimension, raw_dimension)

In [ ]:
pores_flow = {}
for i, j, k in np.transpose(np.where(skimage.segmentation.find_boundaries(pores_img, connectivity=1, mode='inner', background=-1))):
    if i < raw_dimension - 1 and pores_img[i + 1, j, k] != pores_img[i, j, k] and pores_img[i + 1, j, k] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i + 1, j, k])] = pores_flow.get((pores_img[i, j, k], pores_img[i + 1, j, k]), 0) + vz[i, j, k]
    if i > 0 and pores_img[i - 1, j, k] != pores_img[i, j, k] and pores_img[i - 1, j, k] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i - 1, j, k])] = pores_flow.get((pores_img[i, j, k], pores_img[i - 1, j, k]), 0) - vz[i, j, k]
    if j < raw_dimension - 1 and pores_img[i, j + 1, k] != pores_img[i, j, k] and pores_img[i, j + 1, k] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i, j + 1, k])] = pores_flow.get((pores_img[i, j, k], pores_img[i, j + 1, k]), 0) + vy[i, j, k]
    if j > 0 and pores_img[i, j - 1, k] != pores_img[i, j, k] and pores_img[i, j - 1, k] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i, j - 1, k])] = pores_flow.get((pores_img[i, j, k], pores_img[i, j - 1, k]), 0) - vy[i, j, k]
    if k < raw_dimension - 1 and pores_img[i, j, k + 1] != pores_img[i, j, k] and pores_img[i, j, k + 1] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i, j, k + 1])] = pores_flow.get((pores_img[i, j, k], pores_img[i, j, k + 1]), 0) + vx[i, j, k]
    if k > 0 and pores_img[i, j, k - 1] != pores_img[i, j, k] and pores_img[i, j, k - 1] != -1:
        pores_flow[(pores_img[i, j, k], pores_img[i, j, k - 1])] = pores_flow.get((pores_img[i, j, k], pores_img[i, j, k - 1]), 0) - vx[i, j, k]

In [ ]:
np.save(f'../saved_dicts_and_lists/{filename}_pores_flow_{flow_direction}.npy', pores_flow)

## Mean pressure on the halfthroat boundaries

In [ ]:
p = np.fromfile(f'../raw_data/{filename}_{flow_direction}_p.raw', dtype=dtype, sep='')
p = p.reshape(raw_dimension, raw_dimension, raw_dimension)

In [ ]:
halfthroat_img_boundaries = np.fromfile(f'../raw_data/{filename}_halfthroats_boundaries.raw', dtype=np.int32, sep='')
halfthroat_img_boundaries = halfthroat_img_boundaries.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
halfthroat_boundaries_labels = np.unique(halfthroat_img_boundaries)[1:]

In [ ]:
%%time
sum_pressure = scipy.ndimage.sum_labels(p, halfthroat_img_boundaries, halfthroat_boundaries_labels)

In [ ]:
halfthroat_boundaries_indices = scipy.ndimage.value_indices(halfthroat_img_boundaries, ignore_value=0)

In [ ]:
halfthroat_boundaries_pressure = {}
for i in range(len(halfthroat_boundaries_labels)):
    halfthroat_boundaries_pressure[halfthroat_boundaries_labels[i]] = sum_pressure[i] / len(halfthroat_boundaries_indices[halfthroat_boundaries_labels[i]][0])

In [ ]:
np.save(f'../saved_dicts_and_lists/{filename}_halfthroat_boundaries_pressure_{flow_direction}.npy', halfthroat_boundaries_pressure)